# Short Duration Comparison

Flexible short-duration notebook. Build views by genotype, line, cohort, dataset, animal, or custom combinations, prepare the short-duration tables once, then rerun plotting cells without recomputing.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

ROOT

## 2. Choose Datasets

`DATASET_SELECTIONS` is the safest option when mixing lines/cohorts. Set it to `None` to use all combinations of `LINES` and `COHORTS`. Use `ANIMAL_SELECTION` for a one-animal short-duration review.

In [ ]:
LINES = ["CNTNAP2"]
COHORTS = ["cohort3"]

# Explicit selections can mix lines/cohorts, e.g.:
# DATASET_SELECTIONS = [("CNTNAP2", "cohort2"), ("CNTNAP2", "cohort3"), ("SHANK3", "cohort1")]
DATASET_SELECTIONS = None

# Example: "ASD0026" for a single animal, or None for group comparisons.
ANIMAL_SELECTION = "ASD0033"

## 3. Choose Comparison

Common presets:

- Compare genotypes within selected datasets: `COMPARISON = "genotypes"`, `SPLIT_BY = "none"`
- Keep cohorts/datasets separate within genotype: `COMPARISON = "genotypes"`, `SPLIT_BY = "dataset"`
- Compare the same genotype across datasets: `COMPARISON = "datasets"`, `GENOTYPES = ["hom"]`
- One view per animal: `COMPARISON = "animals"`
- Fully custom: `COMPARISON = "custom"` and edit `CUSTOM_SPECS`.

In [ ]:
COMPARISON = "genotypes"  # "genotypes", "datasets", "lines", "cohorts", "animals", or "custom"
SPLIT_BY = "none"         # for COMPARISON="genotypes": "none", "dataset", "line", or "cohort"
GENOTYPES = ["wt", "het", "hom"]

CUSTOM_SPECS = [
    # {"name": "CNTNAP2 hom all cohorts", "line": "CNTNAP2", "genotype": "hom"},
    # {"name": "SHANK3 hom", "line": "SHANK3", "genotype": "hom"},
]

## 4. Load Data And Build Views

Rerun this if you change dataset selection, comparison mode, genotype selection, or custom specs.

In [ ]:
from Pipeline.stimdur import (
    build_stimdur_views,
    build_view_colors,
    build_view_labels,
    load_stimdur_data,
    summarize_views,
)

data = load_stimdur_data(
    lines=LINES,
    cohorts=COHORTS,
    dataset_selections=DATASET_SELECTIONS,
    animal_selection=ANIMAL_SELECTION,
)
df_plot = data["df_plot"]

views = build_stimdur_views(
    df_plot,
    comparison=COMPARISON,
    split_by=SPLIT_BY,
    genotypes=GENOTYPES,
    custom_specs=CUSTOM_SPECS,
)
view_pretty = build_view_labels(views)
view_colors = build_view_colors(views)

print("Loaded datasets:", data["selections"])
print("Usable dataset keys:", data["usable_dataset_names"])
display(summarize_views(df_plot, views))
print("View labels:", view_pretty)
print("View colors:", view_colors)

## 5. Prepare Once

This is the expensive step. Rerun it if you change filters, selected stim durations, views, or datasets. You do not need to rerun it just to change `PLOT_MODE` below.

In [ ]:
from Pipeline.stimdur import DEFAULT_STIM_DURS, prepare_stimdur_comparison
from StimDur.config import FilterConfig, PlotStyle, StimDurComparisonConfig

STIM_DURS = DEFAULT_STIM_DURS

cfg = StimDurComparisonConfig(
    error_mode="individuals",
    skip_psy_fits=(50,),
    ild_shift_for_abl50=True,
)
fcfg = FilterConfig(
    training_min=16,
    session_min=13,
    drop_repeat_trials=True,
    session_type_values=[2],
)
style = PlotStyle(title_fs=24, label_fs=25, tick_fs=24, legend_fs=16)

bundle = prepare_stimdur_comparison(
    df=df_plot,
    views=views,
    stim_durs=STIM_DURS,
    cfg=cfg,
    fcfg=fcfg,
    style=style,
    view_colors=view_colors,
    view_pretty=view_pretty,
)

print("Filtered rows:", len(bundle["df"]))
print("Active stim durations:", [s.name for s in bundle["stimdur_specs"]])

## 6. Plot From Prepared Data

Change `PLOT_MODE` and rerun this cell without recomputing preparation.

- `by_view`: one 4x3 figure per view, lines are stim durations
- `by_stimdur`: one 4x3 figure per stim duration, lines are views
- `performance_by_view`: one 1x3 performance-over-duration figure per view that Alfonso likes
- `performance_all`: one 3x5 performance-over-duration summary for all views
- `kreg_by_view`: one kernel-regression style figure per view
- `all`: make all figure families

In [ ]:
from Pipeline.stimdur import plot_stimdur_comparison

PLOT_MODE = "by_view"

out = plot_stimdur_comparison(
    bundle=bundle,
    views=views,
    plot_mode=PLOT_MODE,
    show=True,
    abls=(20, 40, 60),
    absilds=(1, 2, 4, 8, 16),
)

out["figures"]

## 7. Example Recipes

### Compare genotypes within one cohort

```python
DATASET_SELECTIONS = [("CNTNAP2", "cohort3")]
COMPARISON = "genotypes"
SPLIT_BY = "none"
GENOTYPES = ["wt", "het", "hom"]
```

### Compare genotypes and keep cohorts separate

```python
DATASET_SELECTIONS = [("CNTNAP2", "cohort2"), ("CNTNAP2", "cohort3")]
COMPARISON = "genotypes"
SPLIT_BY = "dataset"
```

### Compare one genotype across datasets

```python
DATASET_SELECTIONS = [("CNTNAP2", "cohort3"), ("SHANK3", "cohort1")]
COMPARISON = "datasets"
GENOTYPES = ["hom"]
```

### Single animal short-duration review

```python
DATASET_SELECTIONS = [("CNTNAP2", "cohort3")]
ANIMAL_SELECTION = "ASD0026"
COMPARISON = "animals"
```